# 📘 Notebook 09: Spark: RDDs and DataFrames

### Course: *From One Table to a Cluster: Working with Data at Scale*
**Instructor:** Ioannis Tsioulis

*Module C: Too Big for One Machine · Notebook 3 of 4 in this module · (9 of 12 overall)*

---

## 🎯 Learning objectives

By the end of this notebook you will be able to:

- Explain what **Apache Spark** adds to the MapReduce model of Notebook 08
- Start a Spark session and create an **RDD**, and say what its **partitions** are
- Distinguish **transformations** from **actions**, and explain **lazy evaluation** with an experiment
- Explain how Spark recovers from failure using an RDD's **lineage**
- Work with **DataFrames**: select, filter, group, aggregate and sort
- Read a **query plan**, and explain why the DataFrame interface is faster than the RDD one
- Use **caching** where it helps

> **Prerequisites:** Notebook 08 (MapReduce: map, shuffle, reduce, combiners, pure functions). Notebooks 02 and 05 (grouping, and column pruning in files).
>
> 🔭 **Why this notebook matters:** MapReduce proved that ordinary programmers could compute on thousands of machines. It also made them write everything as one mapper and one reducer, with the full results saved to disk between every step. Spark, begun as a research project at the University of California, Berkeley in 2009, kept the ideas and removed the pain: chain as many steps as you like, keep intermediate results in memory, and describe the work in a few lines that read almost like SQL. It is today the most widely used engine for processing large data. In this notebook a real copy of it runs inside your notebook, and the code you write is the code that would run, unchanged, on a thousand machines.

> 📦 **Libraries used in this notebook.**
>
> - **PySpark** is the Python interface to **Apache Spark**. Spark itself is written in Scala and runs on the Java virtual machine. PySpark lets you drive it from Python. Here it runs in **local mode**: the processor cores of one machine play the part of the machines of a cluster. That is how Spark programs are normally developed and tested before being sent to a real cluster.
>
> The cell below installs it. In Google Colab, PySpark and the Java it needs are already present and the cell finishes in a moment.

In [ ]:
%pip install -q pyspark

In [ ]:
import csv
import datetime
import os
import random
import tempfile
import time
from operator import add

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

## 1. Starting Spark

Everything in Spark goes through a **session**. Creating one starts the engine, which takes a few seconds the first time.

In [ ]:
spark = (SparkSession.builder
         .master("local[*]")                                   # local mode: use every core of this machine as a worker
         .appName("bookshop")
         .config("spark.sql.shuffle.partitions", "4")          # explained in Section 7
         .config("spark.ui.showConsoleProgress", "false")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
sc = spark.sparkContext

print("Spark version:", spark.version)
print("Workers (cores) available:", sc.defaultParallelism)

The one line that would change on a real cluster is `.master(...)`. In place of `local[*]` it would name the cluster's manager. Nothing else in this notebook would need to be touched.

## 2. The RDD: a collection spread over machines

### Intuition first
In Notebook 08 we cut a list into chunks and pretended each chunk lived on a different machine. Spark's basic object is that idea made real. An **RDD** behaves like a Python list that you can map and filter. Underneath, it is cut into pieces that may live on different machines, and every operation you apply is carried out on all the pieces at once.

### Formal definition
An **RDD**, or *resilient distributed dataset*, is an **immutable** collection of records, divided into **partitions** that can be processed in parallel. *Immutable* means it is never changed: every operation produces a **new** RDD. *Resilient* means it can be rebuilt if a machine is lost, which Section 4 explains.

In [ ]:
lines = [
    "the cat sat on the mat",
    "the dog sat on the log",
    "the cat saw the dog",
    "a dog and a cat",
    "the log and the mat",
    "a cat on a mat",
]

rdd = sc.parallelize(lines, 3)                       # hand the list to Spark, cut into 3 partitions

print("Partitions:", rdd.getNumPartitions())
for number, partition in enumerate(rdd.glom().collect()):        # glom: show the contents of each partition
    print(f"  partition {number}: {partition}")

The same three chunks as in Notebook 08. Now the word count. Compare it with the engine we wrote by hand.

In [ ]:
counts = (rdd.flatMap(lambda line: line.split())     # every line becomes several words
             .map(lambda word: (word, 1))            # the mapper: emit (word, 1)
             .reduceByKey(add)                       # shuffle by key, then add the values up
             .collect())                             # bring the result back to the notebook

print(sorted(counts, key=lambda pair: -pair[1]))

Three operations, chained. `reduceByKey` is the shuffle and the reducer of Notebook 08 in one step, and it applies the function as a **combiner** on each partition first, without being asked.

The difference from MapReduce is that there is no limit of one mapper and one reducer. Steps can be chained freely, as long as you like.

In [ ]:
long_words = (rdd.flatMap(lambda line: line.split())
                 .filter(lambda word: len(word) == 3)          # keep three-letter words
                 .map(lambda word: (word[0], word))            # key: the first letter
                 .groupByKey()                                 # gather the words of each letter
                 .mapValues(lambda words: sorted(set(words)))  # tidy each group
                 .sortByKey()
                 .collect())
print(long_words)

## 3. Transformations, actions and laziness

### Intuition first
You give a builder a list of instructions: knock down this wall, move that door, paint everything white. A poor builder does each as soon as it is spoken, and paints a wall he is about to knock down. A good one **listens to the whole list first**, then works out the best order.

Spark is the good builder. Its operations come in two kinds.

- **Transformations** describe a new RDD in terms of an old one: `map`, `filter`, `flatMap`, `reduceByKey`. Calling one does **no work**. It only adds a step to a plan.
- **Actions** ask for a result: `collect`, `count`, `take`, `sum`. An action makes Spark carry out the whole plan.

Waiting until a result is demanded is called **lazy evaluation**. Let us catch Spark being lazy. The function below is deliberately slow.

In [ ]:
def slow_square(number):
    time.sleep(0.02)                 # pretend this is an expensive computation
    return number * number

numbers = sc.parallelize(range(200), 4)

started = time.perf_counter()
squares = numbers.map(slow_square)                    # a transformation
print(f"Defining the transformation took {time.perf_counter() - started:.4f} seconds")

started = time.perf_counter()
total = squares.sum()                                 # an action
print(f"Running the action took          {time.perf_counter() - started:.2f} seconds   (result: {total:,})")

The transformation returned at once, in no measurable time, although it is supposed to apply a slow function two hundred times. Nothing was computed. All the waiting happened at the second line, when `sum` demanded an answer and Spark finally sent the four partitions to its workers.

Laziness has a practical payoff. Spark sees the whole plan before starting, so it can avoid work that is not needed.

In [ ]:
started = time.perf_counter()
first_three = numbers.map(slow_square).take(3)        # we only want three results
print(f"{first_three} in {time.perf_counter() - started:.2f} seconds")

To produce three results Spark computed only a handful of squares, not all two hundred. An eager system would have computed everything and thrown most of it away.

> ⚠️ **Common pitfalls**
>
> - Believing a slow line is slow where it is written. With lazy evaluation the time is spent at the **action**, which may be many lines after the transformation that causes it.
> - Calling `collect()` on a large RDD. It brings every record back to one machine, the notebook, which is exactly the machine the data did not fit on. Use `take(n)`, or aggregate first.
> - Using `print` inside a function passed to `map` and expecting to see the output. The function runs on the workers, and on a real cluster their output goes to their own logs.

## 4. Lineage: how Spark survives failure

An RDD does not have to store its data. What it always stores is **how it was made**: which RDD it came from, and which operation was applied. This recipe is called its **lineage**. We can ask to see it.

In [ ]:
word_counts = (rdd.flatMap(lambda line: line.split())
                  .map(lambda word: (word, 1))
                  .reduceByKey(add))

print(word_counts.toDebugString().decode())

Read it from the bottom up: a collection was parallelized, then transformed, then shuffled. The indentation marks the **shuffle**, the point where data has to move between machines.

Now recall how MapReduce coped with failure in Notebook 08: it ran the lost chunk again. Spark does the same, with the lineage as its instructions. If a machine dies and a partition is lost, Spark looks up how that partition was derived and **recomputes just that partition** from its source. Nothing needs to have been saved to disk in advance.

This works for the reason it worked in Notebook 08. The functions you pass to `map` and `filter` must be **pure**: the same input must always give the same output, with no other effects. That is also why RDDs are immutable. A dataset that could be changed in place could not be reliably rebuilt.

### Narrow and wide
Some transformations can be done partition by partition, with no communication: `map`, `filter`, `flatMap`. These are called **narrow**. Others need records with the same key to be brought together, whichever partition they are in: `reduceByKey`, `groupByKey`, `sortByKey`, `join`. These are **wide**, and each one causes a **shuffle**.

Shuffles are the expensive part of any Spark job, as they were of any MapReduce job. Spark runs all the narrow steps between two shuffles together, in one pass over each partition. Much of the skill of writing fast Spark programs comes down to shuffling less.

## 5. DataFrames

### Intuition first
An RDD is a collection of arbitrary Python objects. Spark has no idea what is inside them, so it can only run your functions blindly, record by record. Most data, though, is a **table**: rows with named, typed columns. If Spark knows that, it can do far more for us.

### Formal definition
A **DataFrame** is a distributed collection of rows that share a **schema**: a list of named columns with types. It is lazy, immutable and partitioned, like an RDD. Unlike an RDD, its operations are not arbitrary functions. They are expressed in terms of the columns, which lets Spark **understand** the query and plan the best way to run it.

(The word is borrowed from the pandas library, and the idea of a table is the same, but a Spark DataFrame is distributed and lazy, and its methods are different.)

We write the bookshop's order lines to a CSV file, as in Notebook 07, and ask Spark to read it.

In [ ]:
def write_sales(path, lines=200_000, seed=0):
    rng = random.Random(seed)
    cities = ["Athens", "Thessaloniki", "Patras", "Heraklion", "Larissa", "Volos", "Ioannina", "Chania"]
    categories = ["Fiction", "Science", "History", "Children", "Cooking", "Travel", "Art", "Computing"]
    first_day = datetime.date(2023, 1, 1)
    with open(path, "w", newline="", encoding="utf-8") as file:
        writer = csv.writer(file)
        writer.writerow(["line_id", "date", "customer", "city", "category", "quantity", "revenue"])
        for line_id in range(1, lines + 1):
            day = first_day + datetime.timedelta(days=rng.randint(0, 729))
            quantity = rng.randint(1, 3)
            writer.writerow([line_id, day.isoformat(), rng.randint(1, 5000), rng.choice(cities), rng.choice(categories),
                             quantity, round(quantity * rng.uniform(5, 60), 2)])

folder = tempfile.mkdtemp()
sales_path = os.path.join(folder, "sales.csv")
write_sales(sales_path)

sales = spark.read.csv(sales_path, header=True, inferSchema=True)

sales.printSchema()
sales.show(5)
print(f"{sales.count():,} rows in {sales.rdd.getNumPartitions()} partitions")

Spark read the header for the column names and sampled the file to work out the types. The questions of Notebook 02 can now be asked with methods that mirror the clauses of SQL.

| SQL | DataFrame method |
|---|---|
| `SELECT` | `.select(...)` |
| `WHERE` | `.filter(...)` |
| `GROUP BY` ... aggregate | `.groupBy(...).agg(...)` |
| `ORDER BY` | `.orderBy(...)` |
| `LIMIT` | `.limit(...)` |

In [ ]:
(sales.filter(F.col("city") == "Athens")
      .groupBy("category")
      .agg(F.round(F.sum("revenue"), 2).alias("revenue"),
           F.count("*").alias("lines"))
      .orderBy(F.desc("revenue"))
      .show(4))

`F.col("city")` refers to a column, and `F.sum`, `F.count` and `F.round` are Spark's own functions. They are not Python functions that run on each row. They are **descriptions** of what should be computed, which Spark carries out inside its own engine.

New columns are derived with `withColumn`. Here the month is cut out of the date, and revenue is totalled by month.

In [ ]:
by_month = (sales.withColumn("month", F.date_format("date", "yyyy-MM"))
                 .groupBy("month")
                 .agg(F.round(F.sum("revenue")).alias("revenue"), F.countDistinct("customer").alias("customers"))
                 .orderBy("month"))
by_month.show(4)

`by_month` is itself a DataFrame, lazy like everything else. Nothing was computed until `show` asked for the first four rows.

## 6. Why DataFrames are faster: the optimiser

Because a DataFrame query is a description and not a set of opaque functions, Spark can rewrite it before running it. The component that does this is called **Catalyst**, and we can see its decisions with `explain()`, the counterpart of `EXPLAIN QUERY PLAN` in Notebook 03.

In [ ]:
query = (sales.filter(F.col("city") == "Athens")
              .groupBy("category")
              .agg(F.sum("revenue").alias("revenue")))
query.explain()

The plan is read from the bottom upwards. Look at the last line, the scan of the file, and find two phrases.

- `PushedFilters: [... EqualTo(city,Athens)]`. The condition has been handed down to the reader of the file. This is the **predicate pushdown** of Notebook 05.
- `ReadSchema: struct<city:string,category:string,revenue:double>`. Of the seven columns in the file, only the three that the query mentions will be read. This is **column pruning**.

We asked for neither. Spark worked out for itself that the other columns were not needed. Higher up, `HashAggregate` appears twice with an `Exchange` between them: a partial aggregation on each partition, then a shuffle, then the final aggregation. That is the **combiner** of Notebook 08, again applied without being asked.

With an RDD none of this is possible. Spark cannot look inside a Python `lambda` to discover which fields it uses.

There is a second reason for the difference. An RDD of Python objects must be processed by Python, one record at a time, with every record passed between Spark's engine and a Python process. A DataFrame query runs entirely inside Spark's engine. Let us measure the same computation both ways.

In [ ]:
def time_it(function):
    started = time.perf_counter()
    result = function()
    return result, time.perf_counter() - started

rdd_result, rdd_time = time_it(lambda: sales.rdd
                               .map(lambda row: (row["category"], row["revenue"]))
                               .reduceByKey(add)
                               .collect())

frame_result, frame_time = time_it(lambda: sales.groupBy("category").agg(F.sum("revenue")).collect())

print(f"RDD with Python functions: {rdd_time:6.2f} seconds")
print(f"DataFrame:                 {frame_time:6.2f} seconds   ({rdd_time / frame_time:.0f} times faster)")

same = {key: round(value, 2) for key, value in rdd_result} == {row[0]: round(row[1], 2) for row in frame_result}
print("Same result:", same)

The same answer, many times faster, from code that is also shorter and easier to read. This is why nearly all Spark programs today are written with DataFrames or SQL. RDDs remain the foundation underneath, and are still the right tool for data that is not a table at all.

## 7. Partitions, and caching

### How many partitions?
A DataFrame is cut into partitions, and each partition is one unit of work for one core.

In [ ]:
print("Partitions when reading the file:", sales.rdd.getNumPartitions())

grouped = sales.groupBy("category").count()
print("Partitions after a shuffle:      ", grouped.rdd.getNumPartitions())

A shuffle redistributes the rows into a number of partitions set by `spark.sql.shuffle.partitions`. Its default is 200, a sensible number for a cluster working on terabytes. For our small file it would mean 200 nearly empty partitions, each with a fixed cost of scheduling. That is why we set it to 4 when starting the session.

The output shows something further. Recent versions of Spark look at the size of the data **after** the shuffle and merge partitions that turned out to be tiny, which is why you may see fewer than four. This is called *adaptive query execution*: the plan is adjusted while it runs, using what has been learned about the data so far. It is the **small-files problem** of Notebook 05 in another form, with the same lesson: too many pieces, each too small, cost more to manage than to process.

### Caching
Lazy evaluation has one consequence that surprises everyone. A DataFrame is a recipe. Every action follows the recipe again **from the beginning**, including reading the file. If several actions start from the same intermediate result, that result is recomputed for each of them.

When that is wasteful, ask Spark to **keep** the result in memory after computing it the first time.

In [ ]:
athens = sales.filter(F.col("city") == "Athens").withColumn("month", F.date_format("date", "yyyy-MM"))

def three_questions(frame):
    frame.count()
    frame.groupBy("category").agg(F.sum("revenue")).collect()
    frame.groupBy("month").agg(F.avg("quantity")).collect()

nothing, without_cache = time_it(lambda: three_questions(athens))

athens.cache()                  # mark it to be kept
athens.count()                  # the first action computes it and stores it
nothing, with_cache = time_it(lambda: three_questions(athens))

print(f"Three questions, recomputing each time: {without_cache:.2f} seconds")
print(f"Three questions, from the cache:        {with_cache:.2f} seconds")
athens.unpersist()
print("(cache released)")

Caching is itself a trade. The cached data occupies memory on the workers, and memory given to a cache is not available for computation. Cache a result that is expensive to produce **and** used more than once. Caching everything is as much a mistake as caching nothing.

## 8. Where we stand

| | MapReduce (Notebook 08) | Spark RDD | Spark DataFrame |
|---|---|---|---|
| A program is | one mapper and one reducer | any chain of transformations | a chain of column operations, or SQL |
| Between steps, data is | written to disk | kept in memory | kept in memory |
| The system knows about your data | nothing | nothing | its columns and types |
| Optimised automatically | no | no | yes |
| Recovery from failure | run the chunk again | recompute from lineage | recompute from lineage |
| Best for | history | data that is not tabular | almost everything |

Spark kept every principle of the last notebook: split the data, process the pieces independently, shuffle only when records must meet, use pure functions so that lost work can be redone. What it added was a way of **describing** the work at a higher level, so that the machine, and not the programmer, decides how to carry it out. It is the lesson of Notebook 01 once more. Say **what** you want, and let the system work out **how**.

The next notebook follows that thought to its end, by running plain SQL on Spark. It also asks the question this notebook has so far avoided: for data of this size, was a cluster engine the right tool at all?

In [ ]:
spark.stop()                    # shut the engine down and free its memory
print("Spark stopped.")

---
## ✏️ Exercises

*The exercises need a running Spark session. If you ran the `spark.stop()` cell above, run the cell below first to start it again and reload the data.*

In [ ]:
spark = (SparkSession.builder.master("local[*]").appName("bookshop")
         .config("spark.sql.shuffle.partitions", "4")
         .config("spark.ui.showConsoleProgress", "false").getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
sc = spark.sparkContext
sales = spark.read.csv(sales_path, header=True, inferSchema=True)
rdd = sc.parallelize(lines, 3)
print("Spark is running.")

### Exercise 1 (RDD basics)
Using the `rdd` of six lines, compute with RDD operations: (a) the total number of words, (b) the number of distinct words, (c) the longest word or words.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
words = rdd.flatMap(lambda line: line.split())

print("(a)", words.count())
print("(b)", words.distinct().count())
longest = words.map(len).max()
print("(c)", sorted(words.filter(lambda word: len(word) == longest).distinct().collect()))
```

*`words` is used three times. Because RDDs are lazy, the splitting is redone for each action. On six lines that is nothing. On a large dataset it would be a reason to cache `words`.*
</details>

### Exercise 2 (Transformation or action?)
Predict, for each line below, whether it starts any computation. Then run the cell, which times each line, and check your predictions.

In [ ]:
def slow_identity(value):
    time.sleep(0.01)
    return value

data = sc.parallelize(range(100), 4)
steps = [
    ("a = data.map(slow_identity)",       lambda: data.map(slow_identity)),
    ("b = a.filter(lambda x: x % 2 == 0)", lambda: data.map(slow_identity).filter(lambda x: x % 2 == 0)),
    ("b.count()",                          lambda: data.map(slow_identity).filter(lambda x: x % 2 == 0).count()),
    ("b.take(1)",                          lambda: data.map(slow_identity).filter(lambda x: x % 2 == 0).take(1)),
]
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
for description, step in steps:
    started = time.perf_counter()
    step()
    print(f"{description:<36} {time.perf_counter() - started:6.2f} seconds")
```

*The first two are transformations and return at once. `count` must process every record. `take(1)` needs only one result, so Spark looks at a single partition and stops as soon as it has it.*
</details>

### Exercise 3 (DataFrame queries)
With the `sales` DataFrame, find: (a) the five customers who spent the most, with their total, (b) for each city, the number of order lines and the average quantity, sorted by city.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
(sales.groupBy("customer")
      .agg(F.round(F.sum("revenue"), 2).alias("spent"))
      .orderBy(F.desc("spent"))
      .show(5))

(sales.groupBy("city")
      .agg(F.count("*").alias("lines"), F.round(F.avg("quantity"), 3).alias("average_quantity"))
      .orderBy("city")
      .show())
```

*Each query is a single chain that reads in the order the work is done: group, aggregate, sort, show.*
</details>

### Exercise 4 (Read the plan)
Write a DataFrame query for the number of order lines with a quantity of 3 in the category `Travel`, and print its plan with `explain()`. Find in the plan: which columns are read from the file, and which conditions were pushed down to the reader.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
travel_triples = sales.filter((F.col("quantity") == 3) & (F.col("category") == "Travel"))
print(travel_triples.count(), "lines\n")
travel_triples.groupBy().count().explain()
```

*In the scan at the bottom, `ReadSchema` lists only `category` and `quantity`, and `PushedFilters` contains both conditions. Counting rows needs no other column, so none is read. Conditions are combined with `&` and each must be in its own brackets, because of the way Python reads operators.*
</details>

### Exercise 5 (Does caching always help? A little harder)
Take the DataFrame `sales.filter(F.col("category") == "Art")`. Measure the time of (a) one `count()` without caching, (b) calling `.cache()` and then one `count()`, (c) a second `count()` on the cached DataFrame. Explain the three numbers. When would caching have been a waste?

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
art = sales.filter(F.col("category") == "Art")

nothing, plain = time_it(lambda: art.count())
art.cache()
nothing, first = time_it(lambda: art.count())
nothing, second = time_it(lambda: art.count())
art.unpersist()

print(f"(a) without a cache:               {plain:.2f} s")
print(f"(b) first action after cache():    {first:.2f} s")
print(f"(c) second action, from the cache: {second:.2f} s")
```

*The first action after `cache()` is no faster than without it, and is often slower: it does the same work and has to store the result as well. Only later actions benefit. If the DataFrame is used once, caching costs time and memory and returns nothing. It pays when a result is expensive to compute and used several times.*
</details>

In [ ]:
spark.stop()

## 🔑 Key takeaways

- **Spark** keeps the principles of MapReduce and removes its restrictions: any number of chained steps, intermediate results in memory, a high-level interface.
- An **RDD** is an immutable collection split into **partitions** that are processed in parallel.
- **Transformations** build a plan. **Actions** run it. This **lazy evaluation** lets Spark avoid unnecessary work.
- An RDD remembers its **lineage**, and a lost partition is recomputed from it. This requires **pure functions**.
- **Narrow** transformations work within a partition. **Wide** ones cause a **shuffle**, the expensive step.
- A **DataFrame** has named, typed columns. Spark understands DataFrame queries and optimises them, with column pruning, predicate pushdown and partial aggregation. They run much faster than Python functions on an RDD.
- **Cache** a result that is expensive and reused. Set the number of shuffle partitions to suit the size of the data.

---
**Next:** *Notebook 10: Spark SQL and Pipelines*, where the bookshop's whole warehouse moves to Spark, queries are written in plain SQL, and we measure honestly when a cluster engine helps and when it gets in the way.

---
*© Ioannis Tsioulis. *From One Table to a Cluster: Working with Data at Scale*. Prepared for educational use.*